In [ ]:
import pandas as pd
import numpy as np
import cv2
from tqdm import tqdm


# ─────────────────────────────────────────────
# LINE THICKNESS CONTROL (KEY FIX)
# ─────────────────────────────────────────────
THICKNESS_KERNEL = 3   # increase = thicker + more continuous lines
GAP_FILL_KERNEL = 2    # fills tiny breaks






In [ ]:
# ─────────────────────────────────────────────
# LOAD DATA
# ─────────────────────────────────────────────
df = pd.read_csv("data/legend_class.csv")
df = df[df.geometry == "line"].reset_index(drop=True)

img = cv2.imread("data/el_harrach_highres_map.png")
img_i16 = img.astype(np.int16)

tolerance = 1


In [ ]:
# ─────────────────────────────────────────────
# OUTPUT FOLDER
# ─────────────────────────────────────────────
!mkdir -p output/overlays2


In [ ]:
# ─────────────────────────────────────────────
# PROCESS LINES
# ─────────────────────────────────────────────
for i, row in tqdm(df.iterrows(), total=len(df), desc="Generating overlays"):

    hex_color = row["hex"].lstrip("#")
    rgb = np.array([int(hex_color[j:j+2], 16) for j in (0, 2, 4)], dtype=np.int16)
    bgr = rgb[::-1]

    # ─────────────────────────────
    # STEP 1: MASK
    # ─────────────────────────────
    mask = np.all(np.abs(img_i16 - bgr) <= tolerance, axis=2).astype(np.uint8)

    # ─────────────────────────────
    # STEP 2: THICKEN LINES (CRITICAL FIX)
    # replaces broken pixel lines with continuous strokes
    # ─────────────────────────────
    kernel = np.ones((THICKNESS_KERNEL, THICKNESS_KERNEL), np.uint8)
    mask = cv2.dilate(mask, kernel, iterations=1)

    # ─────────────────────────────
    # STEP 3: FILL SMALL GAPS
    # ─────────────────────────────
    kernel2 = np.ones((GAP_FILL_KERNEL, GAP_FILL_KERNEL), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel2)

    # ─────────────────────────────
    # STEP 4: APPLY OVERLAY
    # ─────────────────────────────
    result = img.copy()

    # red overlay (thick + stable)
    result[mask == 1] = (0, 0, 255)

    class_name = row["class"].replace(" ", "_")

    cv2.imwrite(
        f"output/overlays/overlay_{i+1}_{class_name}.png",
        result
    )